In [ ]:
import os
import json
import numpy as np

import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
import pickle

In [ ]:
## CONFIG
SHARD_ID = 9
REPO_ID = f"DSAIL-ReID/synthetic-gzgc"
LOCAL_DIR = Path("synthetic-gzgc-local").resolve()
DATASET_ROOT = LOCAL_DIR
ASSIGNEE = None

In [ ]:
assert ASSIGNEE, 'Enter your name above under "ASSIGNEE"'

In [ ]:
from huggingface_hub import snapshot_download

file_extensions = ['json', 'gif', 'glb', 'jpg'] # 'ply'
patterns = [f'*/shard_{SHARD_ID:03d}/*.{ext}' for ext in file_extensions]

snapshot_download(repo_id="DSAIL-ReID/synthetic-gzgc", repo_type='dataset', local_dir=LOCAL_DIR, allow_patterns=patterns)

In [ ]:
from huggingface_hub import hf_hub_download

shards_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="shards.csv",
    repo_type='dataset',
)

In [ ]:
GIFS_ROOT = DATASET_ROOT / 'renders' / f'shard_{SHARD_ID:03d}' 
gif_paths = list(Path(GIFS_ROOT).rglob('*.gif'))
print(f'Found {len(gif_paths)} gifs')

In [ ]:
MESHES_ROOT = DATASET_ROOT / 'meshes' / f'shard_{SHARD_ID:03d}' 
meshes_paths = list(Path(MESHES_ROOT).rglob('*.glb'))
print(f'Found {len(meshes_paths)} glbs')

In [ ]:
DEEPLABCUT_ROOT = DATASET_ROOT / 'deeplabcut-results' / f'shard_{SHARD_ID:03d}' 
deeplabcut_paths = list(Path(DEEPLABCUT_ROOT).rglob('*.json'))
print(f'Found {len(deeplabcut_paths)} jsons')

In [ ]:
SOURCE_IMAGES_ROOT = DATASET_ROOT / "source-images" / f'shard_{SHARD_ID:03d}'
source_images_paths = list(Path(SOURCE_IMAGES_ROOT).rglob('*.jpg'))
print(f'Found {len(source_images_paths)} jsons')

In [ ]:
keypoint_names = ['nose',
 'upper_jaw',
 'lower_jaw',
 'mouth_end_right',
 'mouth_end_left',
 'right_eye',
 'right_earbase',
 'right_earend',
 'right_antler_base',
 'right_antler_end',
 'left_eye',
 'left_earbase',
 'left_earend',
 'left_antler_base',
 'left_antler_end',
 'neck_base',
 'neck_end',
 'throat_base',
 'throat_end',
 'back_base',
 'back_end',
 'back_middle',
 'tail_base',
 'tail_end',
 'front_left_thai',
 'front_left_knee',
 'front_left_paw',
 'front_right_thai',
 'front_right_knee',
 'front_right_paw',
 'back_left_paw',
 'back_left_thai',
 'back_right_thai',
 'back_left_knee',
 'back_right_knee',
 'back_right_paw',
 'belly_bottom',
 'body_middle_right',
 'body_middle_left']

In [ ]:
gif_paths[:5]

In [ ]:
import pandas as pd
shards_df = pd.read_csv(shards_path)
shard3 = shards_df[shards_df['shard_id']==3]
shard3.head(4)

In [ ]:
gif_path = gif_paths[0]
shard, name = gif_path.parts[-2:]
image_path = SOURCE_IMAGES_ROOT  / (name.split('_')[0] + '.jpg')
image_path, image_path.exists()

In [ ]:
FRAMES_ROOT = Path('output-frames')

output_dir = FRAMES_ROOT / gif_path.stem # os.path.join('output-frames', gif_path.stem)
output_dir.mkdir(parents=True, exist_ok=True) # os.makedirs(output_dir, exist_ok=True)

In [ ]:
shard, name = gif_path.parts[-2:]
json_path = DEEPLABCUT_ROOT / (gif_path.stem+'_superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2_before_adapt.json')
json_path.exists()

In [ ]:
with open(json_path) as f:
    json_out = json.load(f)

In [ ]:
reformated_res = {keypoint_name: {} for keypoint_name in keypoint_names}

for fnum, res in enumerate(json_out):
    for keypoint_name, (x, y, conf) in zip(keypoint_names, res['bodyparts'][0]):
        reformated_res[keypoint_name][fnum] = [x, y, conf]

In [ ]:
for i, keypoint_name in enumerate(keypoint_names):
    keypoint_dict = reformated_res[keypoint_name]

    xs = [i[0] for i in keypoint_dict.values()] # for k,v in keypoint_dict.items()
    ys = [i[1] for i in keypoint_dict.values()] # for k,v in keypoint_dict.items()

    plt.scatter(xs, ys)
    plt.title(f'{i}. {keypoint_name}')
    plt.show()

In [ ]:
for i, keypoint_name in enumerate(keypoint_names):
    keypoint_dict = reformated_res[keypoint_name]

    xs = [i[0] for i in keypoint_dict.values()] # for k,v in keypoint_dict.items()
    ys = [i[1] for i in keypoint_dict.values()] # for k,v in keypoint_dict.items()

    plt.scatter(xs, ys)
    plt.title(f'{i}. {keypoint_name}')
    plt.show()

In [ ]:
lxs = [v[0] for v in reformated_res['left_eye'].values()]
rxs = [v[0] for v in reformated_res['right_eye'].values()]

plt.plot(np.arange(len(lxs)), lxs, label='left_eye', color='blue');

In [ ]:
plt.plot(np.arange(len(rxs)), rxs, label='right_eye', color='green');

In [ ]:
def forward_fill(arr, missing_rep=-1):
    arr = np.array(arr)
    mask = arr == missing_rep
    idx = np.where(~mask, np.arange(len(arr)), 0)
    np.maximum.accumulate(idx, out=idx)
    arr = arr[idx]
    return arr

In [ ]:
import numpy as np

def forward_backward_fill(arr, missing_rep=-1):
    arr = np.array(arr)

    # Mask of missing values
    mask = arr == missing_rep

    # ---------- Forward fill ----------
    idx = np.where(~mask, np.arange(len(arr)), 0)
    np.maximum.accumulate(idx, out=idx)
    arr_ffill = arr[idx]

    # ---------- Backward fill (for leading missings) ----------
    mask_ffill = arr_ffill == missing_rep
    idx_back = np.where(~mask_ffill, np.arange(len(arr_ffill)), len(arr_ffill) - 1)
    idx_back = np.minimum.accumulate(idx_back[::-1])[::-1]
    arr_bfill = arr_ffill[idx_back]

    return arr_bfill

In [ ]:
lxs_ff = forward_fill(lxs)
rxs_ff = forward_fill(rxs)

plt.plot(np.arange(len(rxs_ff)), rxs_ff, label='right_eye', color='green')
plt.plot(np.arange(len(lxs_ff)), lxs_ff, label='left_eye', color='blue')
plt.legend()
plt.show()

In [ ]:
def fold_halfway(arr, image_width=512):
    arr2 = arr.reshape(2,-1).copy()
    arr2[1] = image_width - arr2[1]
    arr2[1] = arr2[1][::-1]
    return arr2

In [ ]:
lxs_ff_folded = np.array([lxs_ff, lxs_ff])
lxs_ff_folded.shape

In [ ]:
# lxs_ff_folded = fold_halfway(lxs_ff)
# rxs_ff_folded = fold_halfway(rxs_ff)

lxs_ff_folded = np.array([lxs_ff, lxs_ff])
rxs_ff_folded = np.array([rxs_ff, rxs_ff])

for eye, eye_arr in zip(['left', 'right'], [lxs_ff_folded, rxs_ff_folded]):
    for i, move in enumerate(['approaching', 'leaving']):
        arr = eye_arr[i]
        plt.plot(np.arange(len(arr)), arr, label=f'{eye}_eye_{move}')
plt.legend()
plt.show()

In [ ]:
from scipy.signal import savgol_filter


def average_and_smooth_v2(left_arr, right_arr, window_size=21):
    arr1, arr2 = left_arr
    arr3, arr4 = right_arr
    smoothed_arrs = [savgol_filter(arr, window_length=window_size, polyorder=3) for arr in [arr1, arr2, arr3, arr4]]
    avg_arr_smooth = np.mean(smoothed_arrs, axis=0)
    return smoothed_arrs, avg_arr_smooth


smoothed_arrs, avg_arr_smooth = average_and_smooth_v2(lxs_ff_folded, rxs_ff_folded)

for i, eye_arr in enumerate(['left', 'right']):
    for j, move in enumerate(['approaching', 'leaving']):
        arr = smoothed_arrs[i*2+j]
        plt.plot(np.arange(len(arr)), arr, label=f'{eye}_eye_{move}')
plt.plot(np.arange(len(avg_arr_smooth)), avg_arr_smooth, label='after smoothing')
plt.legend()
plt.show()

In [ ]:
trough_index = np.argmin(avg_arr_smooth)
trough_index

In [ ]:
left_view = Image.open(gif_path)
left_view.seek(trough_index+20)
left_view

## Loop through Dataset

In [ ]:
OUTPUT_FIGURES_ROOT = Path('output-figures') / f'shard_{SHARD_ID:03d}'
OUTPUT_FIGURES_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt

In [ ]:
lxs_dropped_frames = [i for i in lxs if i==-1]
rxs_dropped_frames = [i for i in rxs if i==-1]

len(lxs_dropped_frames), len(rxs_dropped_frames)

In [ ]:
print(f'dropped frames: lxs: {len(lxs_dropped_frames)}, rxs: {len(rxs_dropped_frames)}')

In [ ]:
# lxs
shift_amnt = np.argmin(np.array(rxs)) - 75
rxs_ = np.roll(rxs, shift_amnt).astype(list)

In [ ]:
trough_indices = {}
dropped_frames_left = {}
dropped_frames_right = {}

In [ ]:
for gif_path in gif_paths:
    # Load JSON predictions
    print(f"\n {'-'*120}\n {gif_path.stem:-^120}")

    output_figures_dir = OUTPUT_FIGURES_ROOT / gif_path.stem
    output_figures_dir.mkdir(parents=True, exist_ok=True)

    shard_foldername = gif_path.parent.name
    json_path = DEEPLABCUT_ROOT / (gif_path.stem+'_superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2_before_adapt.json')
    with open(json_path) as f:
        json_out = json.load(f)

    # Reformat the json for usability
    reformated_res = {keypoint_name: {} for keypoint_name in keypoint_names}

    for fnum, res in enumerate(json_out):
        for keypoint_name, (x, y, conf) in zip(keypoint_names, res['bodyparts'][0]):
            reformated_res[keypoint_name][fnum] = [x, y, conf]

    ## Visualize eye rotations
    lxs = [v[0] for v in reformated_res['left_eye'].values()]
    plt.title('Distance of the left eye from the left edge')
    plt.plot(np.arange(len(lxs)), lxs, label='left_eye', color='blue')
    fig_path = output_figures_dir / '0.png'
    plt.savefig(fig_path)
    plt.show()

    rxs = [v[0] for v in reformated_res['right_eye'].values()]
    plt.title('Distance of the right eye from the left edge')
    plt.plot(np.arange(len(rxs)), rxs, label='right_eye', color='green')
    fig_path = output_figures_dir / '1.png'
    plt.savefig(fig_path)
    plt.show()


    ## Dropped frames
    lxs_dropped_frames = [i for i in lxs if i==-1]
    rxs_dropped_frames = [i for i in rxs if i==-1]
    print(f'dropped frames: lxs: {len(lxs_dropped_frames)}, rxs: {len(rxs_dropped_frames)}')

    ## Forward fill instances where the eye is not visible
    # lxs_ff = forward_fill(lxs)
    # rxs_ff = forward_fill(rxs)

    lxs_ff = forward_backward_fill(lxs)
    rxs_ff = forward_backward_fill(rxs)
    plt.plot(np.arange(len(rxs_ff)), rxs_ff, label='left_eye', color='green')
    plt.plot(np.arange(len(lxs_ff)), lxs_ff, label='left_eye', color='blue')
    plt.title('Eye rotations with forward fill')
    plt.legend()
    fig_path = output_figures_dir / '2.png'
    plt.savefig(fig_path)
    plt.show()

    ## Shift The signal before further processing (...experimental...)
    shift_amnt = -(np.argmin(np.array(rxs_ff)) - 75)
    lxs_ff = np.roll(lxs_ff, shift_amnt) # .astype(list)
    rxs_ff = np.roll(rxs_ff, shift_amnt) # .astype(list)

    plt.plot(np.arange(len(rxs_ff)), rxs_ff, label='left_eye', color='green')
    plt.plot(np.arange(len(lxs_ff)), lxs_ff, label='left_eye', color='blue')
    plt.title('SHIFTED Eye rotations with forward fill')
    plt.legend()
    fig_path = output_figures_dir / '2.5.png'
    plt.savefig(fig_path)
    plt.show()

    ## Fold the eye rotations for averaging
    lxs_ff_folded = fold_halfway(lxs_ff)
    rxs_ff_folded = fold_halfway(rxs_ff)
    # lxs_ff_folded = np.array([lxs_ff, lxs_ff])
    # rxs_ff_folded = np.array([rxs_ff, rxs_ff])

    for eye, eye_arr in zip(['left', 'right'], [lxs_ff_folded, rxs_ff_folded]):
        for i, move in enumerate(['approaching', 'leaving']):
            arr = eye_arr[i]
            plt.plot(np.arange(len(arr)), arr, label=f'{eye}_eye_{move}')
    plt.title('Eye rotations with fold mirroring')
    plt.legend()
    fig_path = output_figures_dir / '3.png'
    plt.savefig(fig_path)
    plt.show()

    ## Averaging and smoothing
    avg_arr, avg_arr_smooth = average_and_smooth_v2(lxs_ff_folded, rxs_ff_folded)
    # trough_index = np.argmin(avg_arr_smooth)
    trough_index = np.argmin(avg_arr_smooth) - shift_amnt + 37
    trough_index = trough_index % 300

    # plt.plot(np.arange(len(avg_arr)), avg_arr, label='raw average')
    # plt.plot(np.arange(len(avg_arr_smooth)), avg_arr_smooth, label='after smoothing')
    # plt.plot(np.arange(avg_arr[0].shape), avg_arr, label='raw average')
    plt.plot(np.arange(len(avg_arr_smooth)), avg_arr_smooth, label='after smoothing')
    plt.vlines(x=3, ymin=2, ymax=8, colors='g', linestyles='solid')
    plt.title('Averaging and smoothing eye rotations')
    plt.legend()
    fig_path = output_figures_dir / '4.png'
    plt.savefig(fig_path)
    plt.show()
    print(f'trough index: {trough_index}')

    ## Show Left Viewpoint Image
    imgg = Image.open(gif_path)
    imgg.seek(trough_index)
    # plt.imshow(imgg)
    # plt.title(f'{gif_path.stem}. Trough index: {trough_index}')
    # plt.axis('off')
    # plt.show()
    src_image_path = SOURCE_IMAGES_ROOT / (gif_path.stem.split('_')[0]+'.jpg')
    src_image = Image.open(src_image_path)
    src_image.thumbnail((300, 300))

    fig, ax = plt.subplots(1, 2) #, figsize=(10, 5))
    ax[0].imshow(src_image)
    ax[1].imshow(imgg)

    ax[0].set_title('Source image')
    ax[1].set_title(f'Left viewpoint. Trough ind: {trough_index}')

    ax[0].axis('off')
    ax[1].axis('off')

    plt.tight_layout()
    fig_path = output_figures_dir / '5.png'
    plt.savefig(fig_path)
    plt.show()

    ## Show Orientation Images
    orientations = ['left', 'back-left', 'back', 'back-right', 'right', 'front-right', 'front', 'front-left']
    fig, ax = plt.subplots(1, len(orientations)+1, figsize=(20, 6))
    ax[0].imshow(src_image)
    ax[0].set_title('Source image')
    ax[0].axis('off')

    for i, orientation in enumerate(orientations):
        steps = int(37.5 * i)
        ind = (trough_index+steps)%300
        imgg.seek(ind)
        ax[i+1].imshow(imgg)
        ax[i+1].set_title(f"{orientation} (ind: {ind})")
        ax[i+1].axis('off')

    plt.tight_layout()
    fig_path = output_figures_dir / '5.png'
    plt.savefig(fig_path)
    plt.show()

    trough_indices[gif_path.name] = trough_index
    dropped_frames_left[gif_path.name] =  len([i for i in lxs if i==-1])
    dropped_frames_right[gif_path.name] =  len([i for i in rxs if i==-1])

## Fiftyone Visualization

In [ ]:
import fiftyone as fo
import fiftyone.brain as fob
import fiftyone.zoo as foz

In [ ]:
groups = [
    'lxs',              # '0.png'
    'rxs',              # '1.png'
    'lxrxs_ff_shifted', # '2.5.png'
    'lrxs_ff',          # '2.0.png'
    'lxrs_fold',        # '3.png'
    'smoothed_average', # '4.png'
    'viewpoints',       # '5.png'
    'threed',           # 3d model
]

In [ ]:
dataset = fo.Dataset(f'shard_{SHARD_ID:03d}-viewpoint-groups', overwrite=True)
dataset.add_group_field("group", default="viewpoints")
dataset.persistent = True

In [ ]:
folders = os.listdir(OUTPUT_FIGURES_ROOT)
samples = []

for folder in folders:
    filenames = sorted(os.listdir(os.path.join(OUTPUT_FIGURES_ROOT, folder)))
    group = fo.Group()
    for group_name, filename in zip(groups, filenames):
        filepath = os.path.join(OUTPUT_FIGURES_ROOT, folder, filename)
        name = filename.split('.')[0]
        # print(name, filepath)
        sample = fo.Sample(filepath=filepath, group=group.element(group_name))
        sample = fo.Sample(
            filepath = filepath,
            group = group.element(group_name),
            dropped_frames_left = dropped_frames_left[f'{folder}.gif'],
            dropped_frames_right = dropped_frames_right[f'{folder}.gif'],
            trough_index = trough_indices[f'{folder}.gif']
            )
        samples.append(sample)

    # load 3d model into group slice
    mesh_path = DATASET_ROOT / 'meshes' / f'shard_{SHARD_ID:03d}' / f'{folder}.glb'
    scene = fo.Scene()
    # scene.camera = fo.PerspectiveCamera(up="Z")
    mesh = fo.GltfMesh("mesh", str(mesh_path), scale=16)
    scene.add(mesh)
    fo3d_path = f"fiftyone-scenes/{folder}.fo3d"
    scene.write(fo3d_path)
    sample = fo.Sample(filepath=fo3d_path, group=group.element("threed"))
    samples.append(sample)

_ = dataset.add_samples(samples)

## Fiftyone Visualization & Sample Tagging

In [ ]:
session = fo.launch_app(dataset, auto=False)

In [ ]:
session.url

In [ ]:
session.show()

In [ ]:
bads = dataset.match_tags('bad')
for sample in bads:
    print(sample.filepath)

In [ ]:
bads_filepaths = [sample.filepath for sample in bads]
bads_filepath_names = [Path(filepath).parent.name for filepath in bads_filepaths]

In [ ]:
df_dict_list = []

for sample in dataset:
    name = Path(sample.filepath).parent.name
    sample['dropped_frames_left']
    sample['dropped_frames_right']

    trough_index = sample['trough_index']
    if name in bads_filepath_names:
        trough_index = -1
    sample_dict = {
        'name': name,
        'dropped_frames_left': sample['dropped_frames_left'],
        'dropped_frames_right': sample['dropped_frames_right'],
        'trough_index': trough_index
    }
    df_dict_list.append(sample_dict)

In [ ]:
df = pd.DataFrame(df_dict_list)
df

In [ ]:
REFERENCES_ROOT = Path('reference-frames')
REFERENCES_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
annots_path = REFERENCES_ROOT / f'shard_{SHARD_ID:03d}.csv'
df.to_csv(annots_path, index=False)

In [ ]:
df

## Upload Completed Verified Reference Frames to Hugging Face

After checking that all the reference cardinal and ordinal directions frames look right, run the code below to upload the csv file containing the information to hugging face


In [ ]:
READY_TO_UPLOAD = False

In [ ]:
if READY_TO_UPLOAD: 
    from huggingface_hub import HfApi
    api = HfApi()
    api.upload_file(
        path_or_fileobj=annots_path,
        path_in_repo=annots_path,
        repo_id=REPO_ID,
        repo_type="dataset",
        commit_message=f'uploading reference frame csv for shard {SHARD_ID} by {ASSIGNEE}'
    )